# Multi-Crisis Signal Dashboard

This notebook implements a phase-based crisis framework:

1. Pre-crisis build-up
2. Cracks appear
3. Systemic break
4. Real-economy spillover

It supports multiple crises using shared indicator definitions and timeline windows.


In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

ROOT = Path('..')

panel = pd.read_csv(ROOT / 'data/processed/indicator_panel_monthly.csv', parse_dates=['date'])
catalog = pd.read_csv(ROOT / 'data/reference/fred_series_catalog.csv')
crises = pd.read_csv(ROOT / 'data/reference/crisis_catalog.csv')
phase_windows = pd.read_csv(ROOT / 'data/reference/crisis_phase_windows.csv', parse_dates=['start_date', 'end_date'])
events = pd.read_csv(ROOT / 'data/reference/timeline_events.csv', parse_dates=['event_date'])

panel.head()

## Crisis Selector
Set `CRISIS_ID` to one of:
- `gfc_2008`
- `dotcom_2000`
- `asian_1997`
- `sl_1980s`


In [ ]:
CRISIS_ID = 'gfc_2008'  # change this value to switch crisis views
USE_ZSCORE = True

phase_order = ['pre_crisis', 'cracks', 'system_break', 'spillover']
phase_labels = {
    'pre_crisis': '1. Pre-crisis build-up',
    'cracks': '2. Cracks appear',
    'system_break': '3. Systemic break',
    'spillover': '4. Real-economy spillover',
}
phase_colors = {
    'pre_crisis': 'rgba(46, 125, 50, 0.10)',
    'cracks': 'rgba(255, 152, 0, 0.11)',
    'system_break': 'rgba(198, 40, 40, 0.12)',
    'spillover': 'rgba(21, 101, 192, 0.10)',
}

value_col = 'value_z' if USE_ZSCORE else 'value'
y_label = 'Normalized value (z-score)' if USE_ZSCORE else 'Raw value'

crisis_row = crises.loc[crises['crisis_id'] == CRISIS_ID].iloc[0]
phase_slice = phase_windows.loc[phase_windows['crisis_id'] == CRISIS_ID].copy()
phase_slice['phase'] = pd.Categorical(phase_slice['phase'], categories=phase_order, ordered=True)
phase_slice = phase_slice.sort_values('phase')

event_slice = events.loc[events['crisis_id'] == CRISIS_ID].copy().sort_values('event_date')

range_start = phase_slice['start_date'].min() - pd.DateOffset(months=6)
range_end = phase_slice['end_date'].max() + pd.DateOffset(months=6)
panel_slice = panel.loc[(panel['date'] >= range_start) & (panel['date'] <= range_end)].copy()

print(f"Crisis: {crisis_row['crisis_name']} ({CRISIS_ID})")
print(f"Date window: {range_start.date()} to {range_end.date()}")

In [ ]:
display(
    phase_slice[['phase', 'start_date', 'end_date']]
    .assign(phase=lambda d: d['phase'].map(phase_labels))
    .rename(columns={'phase': 'Phase', 'start_date': 'Start', 'end_date': 'End'})
)

display(
    event_slice[['event_date', 'event_label', 'event_type']]
    .rename(columns={'event_date': 'Date', 'event_label': 'Event', 'event_type': 'Type'})
)

In [ ]:
def add_phase_context(fig, phase_slice_df, event_slice_df):
    for _, row in phase_slice_df.iterrows():
        phase = row['phase']
        fig.add_vrect(
            x0=row['start_date'],
            x1=row['end_date'],
            fillcolor=phase_colors.get(str(phase), 'rgba(0,0,0,0.06)'),
            layer='below',
            line_width=0,
            annotation_text=phase_labels.get(str(phase), str(phase)),
            annotation_position='top left',
        )

    for _, row in event_slice_df.iterrows():
        fig.add_vline(
            x=row['event_date'],
            line_width=1,
            line_dash='dash',
            line_color='black',
            annotation_text=row['event_label'],
            annotation_position='top right',
        )
    return fig


def plot_dashboard(df, title, color_field='series_id'):
    fig = px.line(
        df,
        x='date',
        y=value_col,
        color=color_field,
        facet_row='domain',
        height=1300,
    )
    fig.update_yaxes(matches=None)
    fig.update_layout(title=title, legend_title_text='Series', margin=dict(l=40, r=40, t=80, b=30))
    add_phase_context(fig, phase_slice, event_slice)
    fig.update_xaxes(title='Date')
    fig.for_each_yaxis(lambda ax: ax.update(title=y_label))
    return fig

## Full Multi-Domain View

In [ ]:
ignore_series = {'USREC'}
df_all = panel_slice.loc[~panel_slice['series_id'].isin(ignore_series)].copy()
plot_dashboard(df_all, f"{crisis_row['crisis_name']} - Signal Dashboard")

## Phase 1: Pre-Crisis Build-up

In [ ]:
pre_ids = catalog.loc[catalog['phase_default'] == 'pre_crisis', 'series_id']
pre = panel_slice.loc[panel_slice['series_id'].isin(pre_ids)]
fig = px.line(pre, x='date', y=value_col, color='series_id', height=450, title='Pre-crisis build-up indicators')
add_phase_context(fig, phase_slice, event_slice)
fig.update_yaxes(title=y_label)
fig.show()

## Phase 2: Cracks Appear

In [ ]:
crack_ids = catalog.loc[catalog['phase_default'] == 'cracks', 'series_id']
cracks = panel_slice.loc[panel_slice['series_id'].isin(crack_ids)]
fig = px.line(cracks, x='date', y=value_col, color='series_id', height=450, title='Cracks appear indicators')
add_phase_context(fig, phase_slice, event_slice)
fig.update_yaxes(title=y_label)
fig.show()

## Phase 3: Systemic Break

In [ ]:
break_ids = catalog.loc[catalog['phase_default'] == 'system_break', 'series_id']
breaks = panel_slice.loc[panel_slice['series_id'].isin(break_ids)]
fig = px.line(breaks, x='date', y=value_col, color='series_id', height=450, title='Systemic break indicators')
add_phase_context(fig, phase_slice, event_slice)
fig.update_yaxes(title=y_label)
fig.show()

## Phase 4: Real-Economy Spillover

In [ ]:
spill_ids = catalog.loc[catalog['phase_default'] == 'spillover', 'series_id']
spill = panel_slice.loc[panel_slice['series_id'].isin(spill_ids)]
fig = px.line(spill, x='date', y=value_col, color='series_id', height=450, title='Real-economy spillover indicators')
add_phase_context(fig, phase_slice, event_slice)
fig.update_yaxes(title=y_label)
fig.show()

## Notes

- `SP500` is available only in recent years from this endpoint, so long-run comparisons rely more on `NASDAQCOM`.
- Keep `USE_ZSCORE = True` for cross-indicator comparisons; use raw values for single-indicator interpretation.
